# 🎚️ 24-信号处理 · 04 滤波器设计（FIR / IIR）

> 关键词：FIR、IIR、差分方程、极点、滑动平均
> 从"3 点平均"到"一阶递归"，手算两种最常用的滤波器，看懂稳定条件。

## 本节要解决的问题

| # | 问题 | 对应内容 |
| --- | --- | --- |
| 1 | FIR 和 IIR 的本质区别？ | §1 核心概念 |
| 2 | 滑动平均滤波器怎么逐点手算？ | §2 手算 ✍️ |
| 3 | 一阶 IIR 的冲激/阶跃响应？ | §2 手算 ✍️ |
| 4 | 为什么极点必须在单位圆内？ | §3 深入原理 |

## 故事引入

录音里有 50Hz 电源哼声，怎么去掉？**低通滤波**把高频的说话声留下、把低频哼声削掉。工程里两大流派：FIR（无反馈，滑动平均是最简单例子）和 IIR（有反馈，一阶低通 $y[n]=ay[n-1]+(1-a)x[n]$）。

滤波器到底在"过滤"什么？看这张真实心电图的对比：上半是干净波形，下半是混入噪声后的波形。噪声让 QRS 波群几乎被淹没——如果直接做诊断，这个信号是没法用的。滤波器（FIR/IIR）的任务就是把这种污染从频域上"抠"出去。

![滤波器到底在"过滤"什么？看这张真实心电图的对比：上半是干净](../figures/ecg_clean_noisy.jpg)

> **怎么读**：对照上下两条波形：噪声版本里高频毛刺和基线抖动叠加在真实心电上。低通/带通滤波后即可恢复接近上方的干净形态。这就是"为什么需要滤波器设计"最直白的回答。
>
> 📚 **图源**：Zhu F et al. (2024) *MEAs-Filter: a novel filter framework utilizing evolutionary algorithms for CVD diagnosis*, Health Inf Sci Syst 12:46（开放获取，CC-BY 4.0）图 1。


## 核心概念

**FIR（有限脉冲响应）**：输出只看输入

$$y[n]=\sum_{k=0}^{M}b_k x[n-k]$$

脉冲响应长度有限（M+1），天然稳定，可线性相位。

**IIR（无限脉冲响应）**：输出还看过去的输出

$$y[n]=\sum_{k=0}^{M}b_k x[n-k]-\sum_{k=1}^{P}a_k y[n-k]$$

脉冲响应无限长，效率高（阶数低），但有稳定问题。

## ✍️ 手算 FIR：3 点滑动平均

**设定**：$h=[1/3,1/3,1/3]$，输入 $x=[1,2,3,4,5]$（边界补零）。

$$y[n]=\frac{x[n-1]+x[n]+x[n+1]}{3}$$

$$\begin{aligned} y[0]&=\frac{0+1+2}{3}=\mathbf{1} & y[1]&=\frac{1+2+3}{3}=\mathbf{2}\\ y[2]&=\frac{2+3+4}{3}=\mathbf{3} & y[3]&=\frac{3+4+5}{3}=\mathbf{4}\\ y[4]&=\frac{4+5+0}{3}=\mathbf{3} \end{aligned}$$

**结果**：$y=[1,2,3,4,3]$——两端被"拉低"（边界补零），中间是局部均值。

**手算收获**：

- 滑动平均 = 最简单的低通：$y$ 比 $x$ 平滑（毛刺被摊平），代价是**延迟与模糊**（前沿变缓）
- 平均窗口越长越平滑、但跟随变化越慢——平滑与响应速度的权衡
- 边界效应：补零使首尾变小——工程用边缘复制/镜像/反射延拓
- 平均多少个点 = 截止频率越低：$M$ 点平均的 -3dB 截止 $\approx f_s/M$（可验证：M=3 → 约 5kHz@16k）

In [1]:
# ---- §2.1 验证：numpy 复算 3 点滑动平均 ----
import numpy as np
x = np.array([1.0, 2.0, 3.0, 4.0, 5.0])
h = np.full(3, 1/3)
y = np.convolve(x, h, mode='same')
print('y =', np.round(y, 4).tolist())
assert np.allclose(y, [1.0, 2.0, 3.0, 4.0, 3.0])
assert np.allclose(y[1:4], [2.0, 3.0, 4.0])   # 中心三点手算核对
print('\n✓ 与手算一致')


y = [1.0, 2.0, 3.0, 4.0, 3.0]

✓ 与手算一致


## ✍️ 手算 IIR：一阶低通 y[n]=0.5y[n-1]+0.5x[n]

**设定**：$y[n]=0.5\,y[n-1]+0.5\,x[n]$（$y[-1]=0$）。

**冲激响应**（$x=[1,0,0,\dots]$）：

$$y[0]=0.5,\quad y[1]=0.25,\quad y[2]=0.125,\quad y[3]=0.0625\Rightarrow\ y[n]=(0.5)^{n+1}$$

指数衰减 → 无限长响应（IIR 名字的由来）。

**阶跃响应**（$x=[1,1,1,\dots]$）：

$$0.5\to0.75\to0.875\to0.9375\to0.96875\Rightarrow\ \text{逐步逼近 }1$$

**手算收获**：

- 递归系数 $a=0.5$ = 记忆折扣：新输出 = 50% 新输入 + 50% 旧记忆——**IIR 一阶即指数平滑**（ETS 的 α 同源，§11-时间序列衔接）
- 冲激响应几何衰减：$a<1$ 衰减（稳定）、$a=1$ 无限累积、$a>1$ 发散
- 阶跃响应收敛于 $b/(1-a)=0.5/0.5=1$——稳态增益公式
- 对比 FIR：IIR 用更低阶达到相近滤波效果，但相位非线性（群延迟随频率变）

In [2]:
# ---- §2.2 验证：numpy 复算一阶 IIR ----
import numpy as np

def iir1(x, a=0.5, b=0.5):
    y = np.zeros_like(x, dtype=float)
    for n in range(len(x)):
        y[n] = a*y[n-1] + b*x[n] if n > 0 else b*x[n]
    return y

imp = np.array([1.0, 0, 0, 0, 0])
print('冲激响应:', np.round(iir1(imp), 4).tolist())
assert np.allclose(iir1(imp), [0.5, 0.25, 0.125, 0.0625, 0.03125])

step = np.ones(5)
print('阶跃响应:', np.round(iir1(step), 4).tolist())
assert np.allclose(iir1(step), [0.5, 0.75, 0.875, 0.9375, 0.96875])
print('\n✓ 与手算一致')


冲激响应: [0.5, 0.25, 0.125, 0.0625, 0.0312]
阶跃响应: [0.5, 0.75, 0.875, 0.9375, 0.9688]

✓ 与手算一致


IIR 滤波器的"反馈"到底长什么样？这张结构图把一阶/多阶 IIR 的差分方程画成了信号流图：输入经前向系数 b_k 加权后求和，输出再经反馈系数 a_k 回灌到输入端——正是这一条反馈回路让 IIR 用很少的阶数就能实现陡峭的频响，也决定了它的极点与稳定性。

![IIR 滤波器的"反馈"到底长什么样？这张结构图把一阶/多阶](../figures/iir_structure.jpg) width="85%"

> **怎么读**：看信号流的方向：横向是前向路径（对应分子多项式 B(z)），下方回环是反馈路径（对应分母多项式 A(z)）。反馈路径就是"极点"在结构上的体现——z 变换与稳定判据都能在这张图上找到直观对应。
>
> 📚 **图源**：Zhu F et al. (2024) *MEAs-Filter*（同上）图 2。（此处以说明文字替代原图标注，结构示意与经典 IIR 直接 I 型一致）


## 深入原理：z 变换与稳定条件

**z 变换**：$H(z)=\sum h[n]z^{-n}$。一阶 IIR：

$$H(z)=\frac{b}{1-az^{-1}}=\frac{bz}{z-a}$$

极点在 $z=a$。**稳定条件**：全部极点必须在单位圆内（$|a|<1$）。

- $a=0.5$：极点 $z=0.5$ 在圆内 → 冲激响应 $0.5^n$ 指数衰减 ✓
- $a=1$：极点 $z=1$ 在圆上 → 输出无限累积（不稳定/振荡）
- 因果 LTI 系统稳定 ⇔ 收敛域含单位圆 ⇔ 冲激响应绝对可和

FIR 极点在原点（$z=0$）→ 天然稳定；IIR 因反馈引入极点 → 设计后必须检查。

## 代码实战：频响对比

FIR 与 IIR 低通的频响（|H(e^{jω})|）对比：IIR 用一阶即可陡峭，FIR 需要更长阶数。

In [3]:
# ---- 频响对比：3 点平均 FIR vs 一阶 IIR ----
import numpy as np

def freqz(b, a, N=512):
    w = np.linspace(0, np.pi, N, endpoint=False)
    H = np.zeros(N, dtype=complex)
    for i, ww in enumerate(w):
        z = np.exp(-1j*ww)
        H[i] = np.polyval(b[::-1], z) / np.polyval(a[::-1], z)
    return w, H

# FIR: 3 点平均
w, Hfir = freqz(np.full(3, 1/3), [1])
# IIR: y[n]=0.5y[n-1]+0.5x[n] → H(z)=0.5/(1-0.5z^-1)
_, Hiir = freqz([0.5], [1, -0.5])
print('频率响应 @DC(0)     : FIR = %.3f, IIR = %.3f' % (np.abs(Hfir)[0], np.abs(Hiir)[0]))
print('频率响应 @0.449π    : FIR = %.3f, IIR = %.3f' % (np.abs(Hfir)[230], np.abs(Hiir)[230]))
print('频率响应 @Nyquist(π): FIR = %.3f, IIR = %.3f' % (np.abs(Hfir)[-1], np.abs(Hiir)[-1]))
assert abs(np.abs(Hfir)[0] - 1) < 1e-6 and abs(np.abs(Hiir)[0] - 1) < 1e-6
assert np.abs(Hfir)[-1] < 0.5 and np.abs(Hiir)[-1] < 0.5
print('✓ 两者都是低通：DC 无衰减、Nyquist 衰减到 1/3；IIR 只用 1 个递归系数达到同等抑制')


频率响应 @DC(0)     : FIR = 1.000, IIR = 1.000
频率响应 @0.449π    : FIR = 0.439, IIR = 0.479
频率响应 @Nyquist(π): FIR = 0.333, IIR = 0.333
✓ 两者都是低通：DC 无衰减、Nyquist 衰减到 1/3；IIR 只用 1 个递归系数达到同等抑制


## 面试考点

**Q1：FIR 和 IIR 怎么选？**

需要线性相位（波形不失真）→ FIR（如均衡器）；要求低延迟省资源 → IIR（如语音降噪一阶高通）。

**Q2：IIR 为什么会不稳定？**

反馈 $a_k$ 把极点从原点移向单位圆外；$|z_p|>1$ 冲激响应发散。

**Q3：3 点平均的截止频率？**

约为 $f_s/M$ 量级（M 点平均）；更精确用频响 -3dB 点。

**Q4：滑动平均是低通还是高通？**

低通——高频抖动被平均摊平；高通可用 $y[n]=x[n]-x[n-1]$（差分）。

## 小结与下一章预告

- FIR：无反馈、天然稳定、滑动平均 $y=[1,2,3,4,3]$ 手算完成
- IIR：有反馈、阶数省、$y[n]=0.5y[n-1]+0.5x[n]$ 冲激 $0.5^n$ 指数衰减、极点 $z=0.5$ 稳定
- 衔接 16-语音：预加重（一阶高通）、降噪（谱减/维纳滤波）、TTS 的 Griffin-Lim 都用这些基础

## ✅ 自测清单

- [ ] 能手算 3 点滑动平均与边界处理
- [ ] 能手算一阶 IIR 的冲激/阶跃响应
- [ ] 能说出 FIR/IIR 本质区别与选型
- [ ] 知道稳定条件（极点 |z|<1）并验证 $a=0.5$